# Preprocessing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle, warnings, os

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')
os.makedirs('../outputs/figures', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

df = pd.read_csv('../data/raw/credit_risk.csv')
print(f'Raw: {df.shape}  |  Missing: {df.isnull().sum().sum()} cells')

## Drop Leaky and Post-Issuance Features

- `loan_percent_income` = `loan_amnt / person_income` — direct target leakage
- `loan_status` — only known after repayment ends, not available at application time

In [ ]:
df_model = df.drop(columns=['loan_percent_income', 'loan_status'])
print('Remaining columns:', list(df_model.columns))

## Impute Missing Values

`loan_int_rate`: per-grade median (rates are grade-dependent, global fill adds noise)  
`person_emp_length`: global median

In [ ]:
# Grade-level medians justify group imputation
grade_medians = df_model.groupby('loan_grade')['loan_int_rate'].median().sort_index()
print('Median interest rate by grade:'); print(grade_medians.round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
grade_medians.plot(kind='bar', ax=axes[0], color='#C62828', edgecolor='white')
axes[0].set_title('Rate varies 8%–21% by grade — global fill would be wrong', fontweight='bold')
axes[0].set_xlabel('Loan Grade'); axes[0].set_ylabel('Median Rate (%)')
axes[0].tick_params(axis='x', rotation=0)

df_model['person_emp_length'].hist(bins=30, ax=axes[1], color='#1565C0', edgecolor='white')
axes[1].axvline(df_model['person_emp_length'].median(), color='red', lw=2, linestyle='--',
                label=f'Median = {df_model["person_emp_length"].median()}')
axes[1].set_title('Employment length', fontweight='bold'); axes[1].legend()
plt.tight_layout()
plt.savefig('../outputs/figures/imputation_rationale.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df_model['person_emp_length'] = df_model['person_emp_length'].fillna(df_model['person_emp_length'].median())
df_model['loan_int_rate']     = df_model['loan_int_rate'].fillna(
    df_model.groupby('loan_grade')['loan_int_rate'].transform('median'))

assert df_model.isnull().sum().sum() == 0
print('No missing values. Shape:', df_model.shape)

## One-Hot Encode Categoricals

No ordinal relationship between categories — `loan_grade` A→G has ordering but `loan_intent` categories don't.

In [ ]:
cat_cols = ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']
df_enc = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)

y = df_enc['loan_amnt']
X = df_enc.drop(columns=['loan_amnt'])
print(f'Features: {X.shape[1]}')
print(list(X.columns))

## Train/Test Split — 80/20

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}')
print(f'Train mean: ${y_train.mean():,.0f}  |  Test mean: ${y_test.mean():,.0f}')

## Scale X and y

Scaling y matters — unscaled MSE on $500–$35k targets runs into the billions and training diverges.

In [ ]:
scaler_X = StandardScaler()
X_train_sc = scaler_X.fit_transform(X_train)
X_test_sc  = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_sc = scaler_y.fit_transform(y_train.values.reshape(-1,1)).flatten()
y_test_sc  = scaler_y.transform(y_test.values.reshape(-1,1)).flatten()

print(f'y_train scaled — mean: {y_train_sc.mean():.3f}, std: {y_train_sc.std():.3f}')
# Example: scaled output 1.3 → dollar amount
print(f'Scaled 1.3 → ${scaler_y.inverse_transform([[1.3]])[0][0]:,.0f}')

## Save

In [ ]:
train_df = pd.DataFrame(X_train_sc, columns=X.columns)
train_df['loan_amnt_scaled'] = y_train_sc
train_df.to_csv('../data/processed/train.csv', index=False)

test_df = pd.DataFrame(X_test_sc, columns=X.columns)
test_df['loan_amnt_scaled'] = y_test_sc
test_df['loan_amnt_raw']    = y_test.values
test_df.to_csv('../data/processed/test.csv', index=False)

with open('../data/processed/scaler_X.pkl','wb') as f: pickle.dump(scaler_X, f)
with open('../data/processed/scaler_y.pkl','wb') as f: pickle.dump(scaler_y, f)
with open('../data/processed/feature_names.pkl','wb') as f: pickle.dump(list(X.columns), f)

print(f'train.csv: {train_df.shape}  |  test.csv: {test_df.shape}')